<a href="https://colab.research.google.com/github/AiWriter404/100-Days-Python/blob/main/real_estate_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real Estate Bot

Converted from `.py` to Jupyter Notebook format.

In [1]:
"""
🏠 REAL ESTATE WHATSAPP BOT
UAE Market | Arabic + English + Urdu
Lead Capture + Property Info + Visit Booking + 7-Day Trial
"""

from flask import Flask, request
from twilio.twiml.messaging_response import MessagingResponse
from twilio.rest import Client
from datetime import datetime, timedelta
import json, os

app = Flask(__name__)

# ══════════════════════════════════════════════
# ✅ CONFIG — Apni info daalo
# ══════════════════════════════════════════════
TWILIO_SID    = "YOUR_TWILIO_SID"
TWILIO_TOKEN  = "YOUR_TWILIO_TOKEN"
TWILIO_NUM    = "whatsapp:+14155238886"

YOUR_WHATSAPP   = "whatsapp:+923001234567"   # Aapka number
PAYMENT_LINK    = "wise.com/pay/yourname"
AGENT_WHATSAPP  = "+971501234567"            # Real estate agent ka number

# ── Trial Settings ──
TRIAL_DAYS  = 7
TRIAL_START = datetime.now()
TRIAL_END   = TRIAL_START + timedelta(days=TRIAL_DAYS)

# ── Agency Info (Client ke liye update karo) ──
AGENCY = {
    "name":      "Al Faris Real Estate",
    "agent":     "Ahmed Al Mansouri",
    "phone":     "+971501234567",
    "email":     "info@alfarisre.ae",
    "location":  "Business Bay, Dubai",
    "areas":     "Dubai Marina, JBR, Downtown, Business Bay, Palm Jumeirah",
    "whatsapp":  "wa.me/971501234567",
}

# ── Lead Storage (simple file-based) ──
LEADS_FILE = "leads.json"

def save_lead(phone, name="", interest="", budget=""):
    leads = []
    if os.path.exists(LEADS_FILE):
        with open(LEADS_FILE, "r") as f:
            leads = json.load(f)
    leads.append({
        "phone": phone,
        "name": name,
        "interest": interest,
        "budget": budget,
        "time": datetime.now().strftime("%Y-%m-%d %H:%M")
    })
    with open(LEADS_FILE, "w") as f:
        json.dump(leads, f, indent=2)

# ══════════════════════════════════════════════
# 🏠 PROPERTY LISTINGS
# ══════════════════════════════════════════════
PROPERTIES = {
    "studio": {
        "en": "🏢 *Studio Apartments Available:*\n• Dubai Marina — AED 45,000/yr\n• JBR — AED 55,000/yr\n• Business Bay — AED 40,000/yr\n\nReply 'visit' to book a viewing! 🗓️",
        "ar": "🏢 *شقق استوديو متاحة:*\n• دبي مارينا — 45,000 درهم/سنة\n• JBR — 55,000 درهم/سنة\n• الخليج التجاري — 40,000 درهم/سنة\n\nرد بـ 'زيارة' لحجز معاينة! 🗓️",
        "ur": "🏢 *اسٹوڈیو اپارٹمنٹ دستیاب:*\n• دبئی مرینا — 45,000 درہم/سال\n• JBR — 55,000 درہم/سال\n• بزنس بے — 40,000 درہم/سال\n\n'وزٹ' لکھیں ویونگ بک کریں! 🗓️"
    },
    "1bhk": {
        "en": "🏠 *1 Bedroom Apartments:*\n• Palm Jumeirah — AED 95,000/yr\n• Dubai Marina — AED 75,000/yr\n• Downtown — AED 85,000/yr\n\nReply 'visit' to book a viewing! 🗓️",
        "ar": "🏠 *شقق غرفة نوم واحدة:*\n• نخلة جميرا — 95,000 درهم/سنة\n• دبي مارينا — 75,000 درهم/سنة\n• وسط المدينة — 85,000 درهم/سنة\n\nرد بـ 'زيارة'! 🗓️",
        "ur": "🏠 *1 بیڈروم اپارٹمنٹ:*\n• پام جمیرہ — 95,000 درہم/سال\n• دبئی مرینا — 75,000 درہم/سال\n• ڈاؤن ٹاؤن — 85,000 درہم/سال\n\n'وزٹ' لکھیں! 🗓️"
    },
    "2bhk": {
        "en": "🏡 *2 Bedroom Apartments:*\n• Dubai Marina — AED 110,000/yr\n• Business Bay — AED 95,000/yr\n• JVC — AED 70,000/yr\n\nReply 'visit' to book! 🗓️",
        "ar": "🏡 *شقق غرفتي نوم:*\n• دبي مارينا — 110,000 درهم/سنة\n• الخليج التجاري — 95,000 درهم/سنة\n• JVC — 70,000 درهم/سنة\n\nرد بـ 'زيارة'! 🗓️",
        "ur": "🏡 *2 بیڈروم اپارٹمنٹ:*\n• دبئی مرینا — 110,000 درہم/سال\n• بزنس بے — 95,000 درہم/سال\n• JVC — 70,000 درہم/سال\n\n'وزٹ' لکھیں! 🗓️"
    },
    "villa": {
        "en": "🏰 *Villas Available:*\n• Palm Jumeirah — AED 350,000/yr\n• Emirates Hills — AED 500,000/yr\n• Arabian Ranches — AED 180,000/yr\n\nCall agent: " + AGENCY['phone'],
        "ar": "🏰 *فيلات متاحة:*\n• نخلة جميرا — 350,000 درهم/سنة\n• تلال الإمارات — 500,000 درهم/سنة\n• المرابع العربية — 180,000 درهم/سنة\n\nاتصل: " + AGENCY['phone'],
        "ur": "🏰 *ولا دستیاب:*\n• پام جمیرہ — 350,000 درہم/سال\n• ایمریٹس ہلز — 500,000 درہم/سال\n\nکال کریں: " + AGENCY['phone']
    }
}

# ══════════════════════════════════════════════
# 💬 REPLIES
# ══════════════════════════════════════════════
REPLIES = {
    "greeting": {
        "en": f"🏠 *Welcome to {AGENCY['name']}!*\n\nI can help you find your perfect property in Dubai!\n\nReply with:\n• *rent* — Rental properties\n• *buy* — Properties for sale\n• *studio* — Studio apartments\n• *1bhk* — 1 Bedroom\n• *2bhk* — 2 Bedrooms\n• *villa* — Villas\n• *visit* — Book a site visit\n• *agent* — Talk to our agent",
        "ar": f"🏠 *مرحباً بكم في {AGENCY['name']}!*\n\nأنا هنا لمساعدتك في إيجاد عقارك المثالي في دبي!\n\nرد بـ:\n• *إيجار* — عقارات للإيجار\n• *شراء* — عقارات للبيع\n• *استوديو* — شقق استوديو\n• *غرفة* — غرفة نوم واحدة\n• *غرفتان* — غرفتا نوم\n• *فيلا* — فيلات\n• *زيارة* — احجز معاينة\n• *وكيل* — تحدث مع وكيلنا",
        "ur": f"🏠 *{AGENCY['name']} میں خوش آمدید!*\n\nدبئی میں آپ کا پرفیکٹ گھر تلاش کرتے ہیں!\n\nجواب دیں:\n• *کرایہ* — کرایے کی پراپرٹی\n• *خریدیں* — خریدنے کے لیے\n• *اسٹوڈیو* — اسٹوڈیو اپارٹمنٹ\n• *1بیڈ* — 1 بیڈروم\n• *2بیڈ* — 2 بیڈروم\n• *ولا* — ولا\n• *وزٹ* — سائٹ وزٹ بک کریں\n• *ایجنٹ* — ایجنٹ سے بات کریں"
    },
    "visit": {
        "en": f"🗓️ *Book a Site Visit*\n\nPlease share:\n1. Your Name\n2. Preferred Date & Time\n3. Property Type\n\nOur agent {AGENCY['agent']} will confirm within 2 hours!\n📞 {AGENCY['phone']}",
        "ar": f"🗓️ *احجز معاينة*\n\nيرجى مشاركة:\n1. اسمك\n2. التاريخ والوقت المفضل\n3. نوع العقار\n\nسيتواصل معك وكيلنا {AGENCY['agent']} خلال ساعتين!\n📞 {AGENCY['phone']}",
        "ur": f"🗓️ *سائٹ وزٹ بک کریں*\n\nبتائیں:\n1. آپ کا نام\n2. پسندیدہ تاریخ و وقت\n3. پراپرٹی کی قسم\n\nہمارے ایجنٹ {AGENCY['agent']} 2 گھنٹے میں تصدیق کریں گے!\n📞 {AGENCY['phone']}"
    },
    "buy": {
        "en": f"💰 *Properties for Sale in Dubai:*\n\n🏢 Studio — from AED 450,000\n🏠 1BR — from AED 750,000\n🏡 2BR — from AED 1,100,000\n🏰 Villa — from AED 2,500,000\n\nAreas: {AGENCY['areas']}\n\nReply 'visit' or call: {AGENCY['phone']}",
        "ar": f"💰 *عقارات للبيع في دبي:*\n\n🏢 استوديو — من 450,000 درهم\n🏠 1 غرفة — من 750,000 درهم\n🏡 2 غرفة — من 1,100,000 درهم\n🏰 فيلا — من 2,500,000 درهم\n\nاتصل: {AGENCY['phone']}",
        "ur": f"💰 *دبئی میں فروخت کے لیے پراپرٹیز:*\n\n🏢 اسٹوڈیو — 450,000 درہم سے\n🏠 1 بیڈ — 750,000 درہم سے\n🏡 2 بیڈ — 1,100,000 درہم سے\n🏰 ولا — 2,500,000 درہم سے\n\nکال: {AGENCY['phone']}"
    },
    "agent": {
        "en": f"👤 *Speak to Our Agent*\n\n{AGENCY['agent']}\n📞 {AGENCY['phone']}\n📧 {AGENCY['email']}\n📍 {AGENCY['location']}\n\nAvailable: Sat–Thu, 9am–7pm",
        "ar": f"👤 *تحدث مع وكيلنا*\n\n{AGENCY['agent']}\n📞 {AGENCY['phone']}\n📧 {AGENCY['email']}\n📍 {AGENCY['location']}\n\nمتاح: السبت–الخميس، 9ص–7م",
        "ur": f"👤 *ایجنٹ سے بات کریں*\n\n{AGENCY['agent']}\n📞 {AGENCY['phone']}\n📧 {AGENCY['email']}\n📍 {AGENCY['location']}\n\nدستیاب: ہفتہ–جمعرات، صبح 9–شام 7"
    },
    "areas": {
        "en": f"📍 *Areas We Cover:*\n\n{AGENCY['areas']}\n\nWhich area interests you? Reply with the area name!",
        "ar": f"📍 *المناطق التي نغطيها:*\n\n{AGENCY['areas']}\n\nأي منطقة تهمك؟",
        "ur": f"📍 *ہم کہاں کام کرتے ہیں:*\n\n{AGENCY['areas']}\n\nکون سا علاقہ پسند ہے؟"
    },
    "default": {
        "en": f"Thank you for your message! 🏠\n\nOur agent will contact you shortly.\n📞 {AGENCY['phone']}\n\nReply *hi* to see all options.",
        "ar": f"شكراً على رسالتك! 🏠\n\nسيتواصل معك وكيلنا قريباً.\n📞 {AGENCY['phone']}",
        "ur": f"آپ کے پیغام کا شکریہ! 🏠\n\nہمارا ایجنٹ جلد رابطہ کرے گا۔\n📞 {AGENCY['phone']}"
    },
    "trial_warning": {
        "en": "⚠️ *TRIAL ENDING IN {days} DAY(S)*\nPay AED 499 to keep your bot active:\n👉 " + PAYMENT_LINK,
        "ar": "⚠️ *تنتهي التجربة خلال {days} يوم*\nادفع 499 درهم للاستمرار:\n👉 " + PAYMENT_LINK,
        "ur": "⚠️ *ٹرائل {days} دن میں ختم*\nجاری رکھنے کے لیے AED 499:\n👉 " + PAYMENT_LINK
    },
    "trial_expired": {
        "en": "❌ Trial ended. Reactivate: " + PAYMENT_LINK,
        "ar": "❌ انتهت التجربة. للتفعيل: " + PAYMENT_LINK,
        "ur": "❌ ٹرائل ختم۔ دوبارہ: " + PAYMENT_LINK
    }
}

# ══════════════════════════════════════════════
# 🌍 LANGUAGE DETECTION
# ══════════════════════════════════════════════
def detect_lang(text):
    arabic = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    urdu_words = ["helo","hello","kya","hai","rent","kiraaya","ghar","kitna",
                  "buy","khareedna","visit","agent","area"]
    if arabic > 2: return "ar"
    if any(w in text.lower() for w in urdu_words): return "ur"
    return "en"

# ══════════════════════════════════════════════
# 🤖 SMART REPLY
# ══════════════════════════════════════════════
def get_reply(msg, lang, user_phone):
    m = msg.lower()

    greet   = ["hi","hello","helo","hey","salam","مرحبا","السلام","ہیلو","start"]
    studio  = ["studio","استوديو","اسٹوڈیو","small","chota"]
    one_bhk = ["1bhk","1 bhk","1bed","1 bed","one bed","غرفة","1بیڈ","1 بیڈ"]
    two_bhk = ["2bhk","2 bhk","2bed","2 bed","two bed","غرفتان","2بیڈ","2 بیڈ"]
    villa_w = ["villa","فيلا","ولا","house","bungalow"]
    buy_w   = ["buy","purchase","sale","خرید","شراء","khareed"]
    rent_w  = ["rent","kiraaya","إيجار","کرایہ","lease"]
    visit_w = ["visit","viewing","book","appointment","زيارة","وزٹ","ملاقات"]
    agent_w = ["agent","human","person","staff","وكيل","ایجنٹ","baat","talk"]
    area_w  = ["area","location","where","kahan","منطقة","علاقہ","marina","jbr","downtown","palm"]

    # Save lead on visit request
    if any(w in m for w in visit_w):
        save_lead(user_phone, interest="site_visit")
        return REPLIES["visit"][lang]

    if any(w in m for w in greet):   return REPLIES["greeting"][lang]
    if any(w in m for w in studio):  return PROPERTIES["studio"][lang]
    if any(w in m for w in one_bhk): return PROPERTIES["1bhk"][lang]
    if any(w in m for w in two_bhk): return PROPERTIES["2bhk"][lang]
    if any(w in m for w in villa_w): return PROPERTIES["villa"][lang]
    if any(w in m for w in buy_w):   return REPLIES["buy"][lang]
    if any(w in m for w in agent_w): return REPLIES["agent"][lang]
    if any(w in m for w in area_w):  return REPLIES["areas"][lang]
    if any(w in m for w in rent_w):  return REPLIES["greeting"][lang]

    save_lead(user_phone, interest="general_inquiry")
    return REPLIES["default"][lang]

# ══════════════════════════════════════════════
# ⏳ TRIAL HELPERS
# ══════════════════════════════════════════════
def is_active():     return datetime.now() < TRIAL_END
def days_left():     return max(0, (TRIAL_END - datetime.now()).days)

# ══════════════════════════════════════════════
# 🤖 MAIN ENDPOINT
# ══════════════════════════════════════════════
@app.route("/realestate", methods=["POST"])
def realestate_bot():
    user_msg   = request.form.get("Body", "").strip()
    user_phone = request.form.get("From", "")
    resp       = MessagingResponse()
    lang       = detect_lang(user_msg)

    if not is_active():
        resp.message(REPLIES["trial_expired"][lang])
        return str(resp)

    base = get_reply(user_msg, lang, user_phone)
    d    = days_left()
    if d <= 2:
        base += "\n\n" + REPLIES["trial_warning"][lang].format(days=d)

    resp.message(base)
    return str(resp)

@app.route("/", methods=["GET"])
def home():
    d = days_left()
    return f"🏠 Real Estate Bot | {'✅ ACTIVE' if is_active() else '❌ EXPIRED'} | {d} days left"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080)


ModuleNotFoundError: No module named 'twilio'